In [ ]:
import numpy as np

# Ejemplo de intercambiar filas
Id = np.eye(4)

print(f'I_orig = {Id}')

Id[[3, 1],:] = Id[[1, 3],:]
print('-------------------')
print(f'I_camb={Id}')

# También se puede hacer para columnas

I_orig = [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
-------------------
I_camb=[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]


In [ ]:
import numpy as np

# Primera versión (siguiendo el algoritmo de Notas del Curso)
def eguassp_1(A,b):
    n = A.shape[0]
    # Salidas
    U = A.copy()
    y = b.copy()

    for k in range(n-1):
        if max(np.abs(U[k:, k]))!=0: # Vemos si vale la pena aplicar Gauss a la columna U[k:, k]
                                     # Si llega a dar cero entonces no hace falta aplicar Gauss ya que
                                     # tenemos la columna nula por debajo de la diagonal principal
                                     # que es lo que buscamos
            l = k + np.argmax((np.abs(U[k:, k]))) # np.argmax permite encontrar el índice del elemento
                                                  # con mayor valor absoluto de un array

            P = np.eye(n)   # Definimos inicialmente a P como una matriz identidad

            P[[k, l], :] = P[[l, k], :]  # Intercambiamos las filas k y l (pivoteo)

            # Seguimos el algoritmo de Notas del Curso
            U = P@U
            y = P@y
            v = U[k+1:, k]/U[k,k]
            U[k+1:, k] = 0
            U[k+1:, k+1:] = U[k+1:, k+1:] - np.outer(v, U[k, k+1:])
            y[k+1:] = y[k+1:] - v*y[k]

    return U, y

# TEST
A = np.random.random((5,5))
b = np.random.random(5)

# Probamos el algoritmo
U, y = eguassp_1(A, b)

print(f'U ={U}')
print('--------------------------------------------------------------------')
print(f'y = {y}')

U =[[ 0.99195345  0.78226645  0.98388902  0.10642395  0.97026484]
 [ 0.          0.39966078 -0.16534599  0.41780023 -0.15823036]
 [ 0.          0.         -0.62019322  0.41626949  0.11492012]
 [ 0.          0.          0.          0.64302834 -0.07565501]
 [ 0.          0.          0.          0.          0.2054905 ]]
--------------------------------------------------------------------
y = [ 0.53325959  0.22051274 -0.0369912   0.52356578 -0.53641582]


In [ ]:
# Segunda versión (Una forma más eficiente para implemnetar el algoritmo)

def eguassp_2(A,b):
    n = A.shape[0]
    # Salidas
    U = A.copy()
    y = b.copy()

    for k in range(n-1):
        if max(np.abs(U[k:, k]))!=0: # Vemos si vale la pena aplicar Gauss a la columna U[k:, k]
                                     # Si llega a dar cero entonces no hace falta aplicar Gauss ya que
                                     # tenemos la columna nula por debajo de la diagonal principal
                                     # que es lo que queremos

            l = k + np.argmax((np.abs(U[k:, k]))) # np.argmax permite encontrar el índice del elemento
                                                  # con mayor valor absoluto de un array

            U[[k, l], :] = U[[l, k], :] # Directamente hacemos el intercambio de filas en U e y
            y[[k, l]] = y[[l, k]]       # Este intercambio no tiene un costo computacional
                                        # Nos evita realizar operaciones de producto en punto flotante
                                        # como por ejemplo P@U y P@y.

            # A partir de aquí el algoritmo se repite igual que el de egauss_1
            v = U[k+1:, k]/U[k,k]
            U[k+1:, k] = 0
            U[k+1:, k+1:] = U[k+1:, k+1:] - np.outer(v, U[k, k+1:])
            y[k+1:] = y[k+1:] - v*y[k]

    return U, y

# TEST
A = np.random.random((5,5))
b = np.random.random(5)

U, y = eguassp_1(A, b)

print(f'U ={U}')
print('--------------------------------------------------------------------')
print(f'y = {y}')

U =[[ 0.85027137  0.69365724  0.18419832  0.34765881  0.6833268 ]
 [ 0.          0.21854508  0.30490717  0.24199187 -0.19648173]
 [ 0.          0.          0.86843963  0.13220188  0.31311412]
 [ 0.          0.          0.          0.3799007   0.46867523]
 [ 0.          0.          0.          0.          0.05795525]]
--------------------------------------------------------------------
y = [ 0.06050307  0.46628412  0.88973696 -0.0604335   0.24753321]


In [ ]:
# Descomposición LU con permutaciones

def dlup(A):
    n = A.shape[0]
    U = A.copy()
    P = np.eye(n)

    for k in range(n-1):
        if max(np.abs(U[k:, k]))!=0: # Vemos si vale la pena aplicar Gauss a la columna U[k:, k]
                                     # Si llega a dar cero entonces no hace falta aplicar Gauss ya que
                                     # tenemos la columna nula por debajo de la diagonal principal
                                     # que es lo que queremos

            l = k + np.argmax((np.abs(U[k:, k]))) # np.argmax permite encontrar el índice del elemento
                                                  # con mayor valor absoluto de un array

            if l !=k: #Faltaba esta condición para evitar permutaciones inecesarias
              U[[k, l], :] = U[[l, k], :] # Permutamos las filas k y l (pivoteo)
              P[[k, l], :] = P[[l, k], :]

            U[k+1:, k] = U[k+1:, k]/U[k,k]
            U[k+1:, k+1:] = U[k+1:, k+1:] - np.outer(U[k+1:, k], U[k, k+1:])

    L = np.tril(U, -1) + np.eye(n) # L se encuentra en la parte inferior de U sin la diagonal
    U = np.triu(U) # U es solo la parte triangular superior

    return L, U, P

# TEST
A = np.random.random((5,5))
L, U, P = dlup(A)

print(f'L = {L}')
print('-------------------------------------------------------')
print(f'U = {U}')
print('-------------------------------------------------------')
print(f'P = {P}')
print('-------------------------------------------------------')
print(f'LU - PA = {L@U-P@A}')

L = [[ 1.          0.          0.          0.          0.        ]
 [ 0.19110398  1.          0.          0.          0.        ]
 [ 0.4584303  -0.59028473  1.          0.          0.        ]
 [ 0.25771388 -0.2086247   0.63155673  1.          0.        ]
 [ 0.33930001  0.0376047   0.41938894  0.46078466  1.        ]]
-------------------------------------------------------
U = [[ 0.89329901  0.79935904  0.27554598  0.70634572  0.98460222]
 [ 0.          0.50957216  0.59746855  0.1139705   0.26282224]
 [ 0.          0.          0.68059583  0.34331464  0.45566597]
 [ 0.          0.          0.          0.53047278  0.38744839]
 [ 0.          0.          0.          0.         -0.22024858]]
-------------------------------------------------------
P = [[0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]]
-------------------------------------------------------
LU - PA = [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [